In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/data/VIN/ML/EL4TF


In [19]:
import numpy as np
# from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

from loaders._load_vn30_multi_label import preprocess
from loaders._load_vn30_meta import VN30, TARGETS
from joblib import dump

In [18]:
import os
os.makedirs('checkpoints/svm', exist_ok=True)

In [3]:

def validate_and_clean_data(X_train, X_test, y_train, y_test, symbol):
    """
    Validate and clean data to ensure it's suitable for training
    """
    print(f"Validating data for {symbol}...")
    
    # Convert pandas DataFrames to numpy arrays if needed
    if hasattr(X_train, 'values'):
        X_train = X_train.values
    if hasattr(y_train, 'values'):
        y_train = y_train.values
    if hasattr(X_test, 'values'):
        X_test = X_test.values
    if hasattr(y_test, 'values'):
        y_test = y_test.values
    
    print(f"Data types - X_train: {type(X_train)}, y_train: {type(y_train)}")
    print(f"Data types - X_test: {type(X_test)}, y_test: {type(y_test)}")
    
    # Check for infinity values
    if np.any(np.isinf(X_train)) or np.any(np.isinf(X_test)):
        print(f"Warning: Found infinity values in data for {symbol}")
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e10, neginf=-1e10)
        X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e10, neginf=-1e10)
    
    # Check for very large values
    if np.any(np.abs(X_train) > 1e10) or np.any(np.abs(X_test) > 1e10):
        print(f"Warning: Found very large values in data for {symbol}")
        X_train = np.clip(X_train, -1e10, 1e10)
        X_test = np.clip(X_test, -1e10, 1e10)
    
    # Check for NaN values
    if np.any(np.isnan(X_train)) or np.any(np.isnan(X_test)):
        print(f"Warning: Found NaN values in data for {symbol}")
        X_train = np.nan_to_num(X_train, nan=0.0)
        X_test = np.nan_to_num(X_test, nan=0.0)
    
    # Ensure data is float64 for features and int64 for labels
    X_train = X_train.astype(np.float64)
    X_test = X_test.astype(np.float64)
    y_train = y_train.astype(np.int64)
    y_test = y_test.astype(np.int64)
    
    print(f"After cleaning - X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
    print(f"After cleaning - y_train shape: {y_train.shape}, dtype: {y_train.dtype}")
    
    # Final validation
    if np.any(np.isnan(X_train)) or np.any(np.isnan(X_test)):
        raise ValueError(f"Still contains NaN values after cleaning for {symbol}")
    
    if np.any(np.isinf(X_train)) or np.any(np.isinf(X_test)):
        raise ValueError(f"Still contains infinity values after cleaning for {symbol}")
    
    return X_train, X_test, y_train, y_test

In [20]:
# Dictionary to store results for all symbols
all_results = {}

# Process each symbol
for symbol in VN30:
    print(f"\n{'='*50}")
    print(f"Processing symbol: {symbol}")
    print(f"{'='*50}")
    
    try:
        # Load data for multi-label classification
        data = preprocess(symbol=symbol, lag=30, verbose=True)
        
        # Extract data splits
        X_train, y_train = data["train"]
        X_test, y_test = data["test"]
        scaler = data["scaler"]
        feature_names = data["feature_names"]
        label_names = data["label_names"]
        
        print(f"Training data shape: {X_train.shape}")
        print(f"Training labels shape: {y_train.shape}")
        print(f"Feature names: {len(feature_names)}")
        print(f"Label names: {label_names}")
        
        # Validate and clean data
        X_train, X_test, y_train, y_test = validate_and_clean_data(
            X_train, X_test, y_train, y_test, symbol
        )
        
        base_svc = SVC(
            kernel='sigmoid',
            C=0.1,
            gamma='scale',
        )
        
        # Wrap with MultiOutputClassifier for multi-label support
        model = MultiOutputClassifier(base_svc)
        
        print("Training Random Forest model with MultiOutputClassifier...")
        model.fit(X_train, y_train)
        dump(model, f'checkpoints/svm/{symbol}_model.joblib')
        
        # Make predictions
        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)
        
        # Calculate metrics for each label separately
        train_accuracies = []
        test_accuracies = []
        
        for i, label_name in enumerate(label_names):
            train_acc = balanced_accuracy_score(y_train[:, i], train_preds[:, i])
            test_acc = balanced_accuracy_score(y_test[:, i], test_preds[:, i])
            train_accuracies.append(train_acc)
            test_accuracies.append(test_acc)
            print(f"{label_name}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")
        
        # Average accuracy across all labels
        avg_train_acc = np.mean(train_accuracies)
        avg_test_acc = np.mean(test_accuracies)
        
        print(f"Average Training Accuracy: {avg_train_acc:.4f}")
        print(f"Average Test Accuracy: {avg_test_acc:.4f}")
        
        # Store results
        all_results[symbol] = {
            'train_accuracy': avg_train_acc,
            'test_accuracy': avg_test_acc,
            'train_predictions': train_preds,
            'test_predictions': test_preds,
            'model': model,
            'label_accuracies': dict(zip(label_names, test_accuracies))
        }
        
        # # Feature importance (average across all outputs)
        # feature_importance = np.mean([estimator.feature_importances_ for estimator in model.estimators_], axis=0)
        # feature_importance_df = pd.DataFrame({
        #     'feature': feature_names,
        #     'importance': feature_importance
        # }).sort_values('importance', ascending=False)
        
        # print(f"\nTop 10 most important features:")
        # print(feature_importance_df.head(10))
        
    except Exception as e:
        print(f"Error processing {symbol}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue


Processing symbol: ACB
Original train shape: (1258, 9)
Original test shape: (315, 9)
Using lag: 30
After feature creation - Train shape: (1227, 48)
After feature creation - Test shape: (284, 48)
Final splits:
  Train: (981, 39)
  Val: (246, 39)
  Test: (284, 39)
  Labels: (981, 3)
  Feature names: 39
Training data shape: (981, 39)
Training labels shape: (981, 3)
Feature names: 39
Label names: ['Up', 'HighVol', 'BreakMA5']
Validating data for ACB...
Data types - X_train: <class 'numpy.ndarray'>, y_train: <class 'numpy.ndarray'>
Data types - X_test: <class 'numpy.ndarray'>, y_test: <class 'numpy.ndarray'>
After cleaning - X_train shape: (981, 39), dtype: float64
After cleaning - y_train shape: (981, 3), dtype: int64
Training Random Forest model with MultiOutputClassifier...
Up: Train Acc=0.8690, Test Acc=0.8281
HighVol: Train Acc=0.7785, Test Acc=0.7922
BreakMA5: Train Acc=0.8830, Test Acc=0.8592
Average Training Accuracy: 0.8435
Average Test Accuracy: 0.8265

Processing symbol: BCM
Ori

In [21]:
# Summary of all results
print(f"\n{'='*60}")
print("SUMMARY OF ALL SYMBOLS")
print(f"{'='*60}")

if all_results:
    results_summary = []
    for symbol, results in all_results.items():
        results_summary.append({
            'Symbol': symbol,
            'Train_Accuracy': np.round(results['train_accuracy'], 4),
            'Test_Accuracy': np.round(results['test_accuracy'], 4)
        })
    
    summary_df = pd.DataFrame(results_summary)
    print(summary_df)
    
    # Overall performance
    avg_train_acc = summary_df['Train_Accuracy'].mean()
    avg_test_acc = summary_df['Test_Accuracy'].mean()
    print(f"\nAverage Training Accuracy: {avg_train_acc:.4f}")
    print(f"Average Test Accuracy: {avg_test_acc:.4f}")
    
    print(f"\nProcessing completed for {len(all_results)} symbols.")
    
    # Save results to file
    summary_df.to_csv('vn30_svm_results.csv', index=False)
    print("Results saved to 'vn30_svm_results.csv'")
    
else:
    print("No symbols were processed successfully. Please check the errors above.")



SUMMARY OF ALL SYMBOLS
   Symbol  Train_Accuracy  Test_Accuracy
0     ACB          0.8435         0.8265
1     BCM          0.8218         0.8116
2     BID          0.8659         0.8215
3     BVH          0.8572         0.8277
4     CTG          0.8571         0.7919
5     FPT          0.8496         0.7526
6     GAS          0.8550         0.8423
7     GVR          0.8990         0.9079
8     HDB          0.8626         0.7483
9     HPG          0.8722         0.8482
10    LPB          0.8731         0.6148
11    MBB          0.8577         0.8379
12    MSN          0.8641         0.8555
13    MWG          0.8758         0.8385
14    PLX          0.8423         0.8629
15    SAB          0.8036         0.6713
16    SHB          0.8351         0.7484
17    SSB          0.7438         0.7114
18    SSI          0.8777         0.8389
19    STB          0.8932         0.8474
20    TCB          0.8488         0.8122
21    TPB          0.8608         0.8478
22    VCB          0.8565        

In [26]:
from sklearn.model_selection import GridSearchCV

# Dictionary to store results for all symbols
all_results = {}

# Candidate hyperparameters
param_grid = {
    'C': [0.01, 0.1, 1, 3, 10, 30],
    'kernel': ['linear', 'rbf', 'sigmoid', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3, 4]  # Only relevant for 'poly' kernel
}

best_params_global = None

# Process each symbol
for idx, symbol in enumerate(VN30):
    print(f"\n{'='*50}")
    print(f"Processing symbol: {symbol}")
    print(f"{'='*50}")
    
    try:
        # Load data for multi-label classification
        data = preprocess(symbol=symbol, lag=30, verbose=True)
        
        # Extract data splits
        X_train, y_train = data["train"]
        X_test, y_test = data["test"]
        scaler = data["scaler"]
        feature_names = data["feature_names"]
        label_names = data["label_names"]
        
        print(f"Training data shape: {X_train.shape}")
        print(f"Training labels shape: {y_train.shape}")
        print(f"Feature names: {len(feature_names)}")
        print(f"Label names: {label_names}")
        
        # Validate and clean data
        X_train, X_test, y_train, y_test = validate_and_clean_data(
            X_train, X_test, y_train, y_test, symbol
        )
        
        # ===============================
        # GridSearch only for the FIRST symbol
        # ===============================
        if idx == 0:
            print("\nRunning GridSearchCV on the first symbol...")
            svc = SVC()
            grid_search = GridSearchCV(
                estimator=svc,
                param_grid=param_grid,
                scoring='balanced_accuracy',
                cv=3,
                n_jobs=-1,
                verbose=1
            )
            
            grid_search.fit(X_train, y_train[:, 0])  # label đầu tiên trong symbol này
            best_params_global = grid_search.best_params_
            print(f"Best params (from first symbol): {best_params_global}")
        
        # ===============================
        # Train MultiOutputClassifier với best params tìm được
        # ===============================
        base_svc = SVC(**best_params_global)
        model = MultiOutputClassifier(base_svc)
        
        print("Training MultiOutputClassifier with best params...")
        model.fit(X_train, y_train)
        dump(model, f'checkpoints/svm/{symbol}_model.joblib')
        
        # Make predictions
        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)
        
        # Calculate metrics for each label separately
        train_accuracies = []
        test_accuracies = []
        
        for i, label_name in enumerate(label_names):
            train_acc = balanced_accuracy_score(y_train[:, i], train_preds[:, i])
            test_acc = balanced_accuracy_score(y_test[:, i], test_preds[:, i])
            train_accuracies.append(train_acc)
            test_accuracies.append(test_acc)
            print(f"{label_name}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")
        
        # Average accuracy across all labels
        avg_train_acc = np.mean(train_accuracies)
        avg_test_acc = np.mean(test_accuracies)
        
        print(f"Average Training Accuracy: {avg_train_acc:.4f}")
        print(f"Average Test Accuracy: {avg_test_acc:.4f}")
        
        # Store results
        all_results[symbol] = {
            'train_accuracy': avg_train_acc,
            'test_accuracy': avg_test_acc,
            'train_predictions': train_preds,
            'test_predictions': test_preds,
            'model': model,
            'label_accuracies': dict(zip(label_names, test_accuracies)),
            'used_params': best_params_global
        }
        
    except Exception as e:
        print(f"Error processing {symbol}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue



Processing symbol: ACB
Original train shape: (1258, 9)
Original test shape: (315, 9)
Using lag: 30
After feature creation - Train shape: (1227, 48)
After feature creation - Test shape: (284, 48)
Final splits:
  Train: (981, 39)
  Val: (246, 39)
  Test: (284, 39)
  Labels: (981, 3)
  Feature names: 39
Training data shape: (981, 39)
Training labels shape: (981, 3)
Feature names: 39
Label names: ['Up', 'HighVol', 'BreakMA5']
Validating data for ACB...
Data types - X_train: <class 'numpy.ndarray'>, y_train: <class 'numpy.ndarray'>
Data types - X_test: <class 'numpy.ndarray'>, y_test: <class 'numpy.ndarray'>
After cleaning - X_train shape: (981, 39), dtype: float64
After cleaning - y_train shape: (981, 3), dtype: int64

Running GridSearchCV on the first symbol...
Fitting 3 folds for each of 144 candidates, totalling 432 fits
Best params (from first symbol): {'C': 3, 'degree': 2, 'gamma': 'scale', 'kernel': 'linear'}
Training MultiOutputClassifier with best params...
Up: Train Acc=1.0000, T

In [27]:

# Summary of all results
print(f"\n{'='*60}")
print("SUMMARY OF ALL SYMBOLS")
print(f"{'='*60}")

if all_results:
    results_summary = []
    for symbol, results in all_results.items():
        results_summary.append({
            'Symbol': symbol,
            'Train_Accuracy': np.round(results['train_accuracy'], 4),
            'Test_Accuracy': np.round(results['test_accuracy'], 4),

        })
    
    summary_df = pd.DataFrame(results_summary)
    print(summary_df)
    
    # Overall performance
    avg_train_acc = summary_df['Train_Accuracy'].mean()
    avg_test_acc = summary_df['Test_Accuracy'].mean()
    print(f"\nAverage Training Accuracy: {avg_train_acc:.4f}")
    print(f"Average Test Accuracy: {avg_test_acc:.4f}")
    
    print(f"\nProcessing completed for {len(all_results)} symbols.")
    
    # Save results to file
    summary_df.to_csv('vn30_svm_results.csv', index=False)
    print("Results saved to 'vn30_svm_results.csv'")
    
else:
    print("No symbols were processed successfully. Please check the errors above.")



SUMMARY OF ALL SYMBOLS
   Symbol  Train_Accuracy  Test_Accuracy
0     ACB          0.9954         0.9900
1     BCM          0.9922         0.9568
2     BID          0.9947         0.9885
3     BVH          0.9936         0.9868
4     CTG          0.9981         0.9841
5     FPT          0.9958         0.9578
6     GAS          0.9980         0.9708
7     GVR          0.9946         0.9792
8     HDB          0.9930         0.9735
9     HPG          0.9957         0.9746
10    LPB          0.9968         0.9781
11    MBB          0.9956         0.9715
12    MSN          0.9939         0.9231
13    MWG          0.9958         0.9513
14    PLX          0.9950         0.9842
15    SAB          0.9922         0.9595
16    SHB          0.9942         0.9546
17    SSB          0.9933         0.9634
18    SSI          0.9950         0.9780
19    STB          0.9958         0.9898
20    TCB          0.9966         0.9685
21    TPB          0.9942         0.9733
22    VCB          0.9961        